
# ExoMinerals — Phase 3: First Pass at the Modeling Approach (Supervised Proxy)

This part is the **first aspect of the modeling approach** from our Mapping plan:
- **Supervised learning proxy** - uses a small **Solar System reference set** to learn weights mapping astrophysical/planetary features to mineral group likelihoods.
- Applies those learned weights to **exoplanets** from the NASA Exoplanet Archive PS table (`PS.csv`).



## 1. Imports


In [4]:

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score

import matplotlib.pyplot as plt

# Display options
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)



## 2. Load Exoplanet PS table


In [5]:

file_path = "PS.csv"

df = pd.read_csv(
    file_path,
    comment="#",    # skip metadata lines starting with #
    sep=",",        # true CSV after header
    engine="python"
)

print(df.shape)
df.head()


(38898, 122)


,rowid,pl_name,hostname,pl_letter,hd_name,hip_name,tic_id,gaia_id,default_flag,sy_snum,sy_pnum,sy_mnum,cb_flag,discoverymethod,disc_year,disc_refname,disc_pubdate,disc_locale,disc_facility,disc_telescope,disc_instrument,rv_flag,pul_flag,ptv_flag,tran_flag,ast_flag,obm_flag,micro_flag,etv_flag,ima_flag,dkin_flag,soltype,pl_controv_flag,pl_refname,pl_orbper,pl_orbsmax,pl_rade,pl_radj,pl_masse,pl_massj,pl_msinie,pl_msinij,pl_cmasse,pl_cmassj,pl_bmasse,pl_bmassj,pl_bmassprov,pl_dens,pl_orbeccen,pl_insol,pl_eqt,pl_orbincl,pl_tranmid,pl_tsystemref,ttv_flag,pl_imppar,pl_trandep,pl_trandur,pl_ratdor,pl_ratror,...,pl_orblper,pl_rvamp,pl_projobliq,pl_trueobliq,st_refname,st_spectype,st_teff,st_rad,st_mass,st_met,st_metratio,st_lum,st_logg,st_age,st_dens,st_vsin,st_rotp,st_radv,sy_refname,rastr,ra,decstr,dec,glat,glon,elat,elon,sy_pm,sy_pmra,sy_pmdec,sy_dist,sy_plx,sy_bmag,sy_vmag,sy_jmag,sy_hmag,sy_kmag,sy_umag,sy_gmag,sy_rmag,sy_imag,sy_zmag,sy_w1mag,sy_w2mag,sy_w3mag,sy_w4mag,sy_gaiamag,sy_icmag,sy_tmag,sy_kepmag,rowupdate,pl_pubdate,releasedate,pl_nnotes,st_nphot,st_nrvc,st_nspec,pl_nespec,pl_ntranspec,pl_ndispec
0,1,11 Com b,11 Com,b,HD 107383,HIP 60202,TIC 72437047,Gaia DR2 3946945413106333696,1,2,1,0,0,Radial Velocity,2007,<a refstr=LIU_ET_AL__2008 href=https://ui.adsa...,2008-01,Ground,Xinglong Station,2.16 m Telescope,Coude Echelle Spectrograph,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Published Confirmed,0.0,<a refstr=TENG_ET_AL__2023 href=https://ui.ads...,323.21,1.178,NaN,NaN,NaN,NaN,4914.898486,15.464,NaN,NaN,4914.898486,15.464,Msini,NaN,0.238,NaN,NaN,NaN,NaN,JD,0.0,NaN,NaN,NaN,NaN,NaN,...,91.33,288.63,NaN,NaN,<a refstr=TENG_ET_AL__2023 href=https://ui.ads...,G8 III,4874.0,13.76,2.09,-0.26,[Fe/H],1.97823,2.45,NaN,NaN,NaN,NaN,NaN,<a refstr=STASSUN_ET_AL__2019 href=https://ui....,12h20m42.91s,185.178779,+17d47m35.71s,17.793252,78.28058,264.13775,18.33392,177.41790,140.383627,-109.24100,88.1701,93.1846,10.71040,5.726,4.72307,2.943,2.484,2.282,NaN,NaN,NaN,NaN,NaN,0.639,0.732,2.358,2.270,4.44038,NaN,3.83790,NaN,9/19/2023,2023-08,9/19/2023,2.0,1.0,2.0,0.0,0.0,0.0,0.0
1,2,11 Com b,11 Com,b,HD 107383,HIP 60202,TIC 72437047,Gaia DR2 3946945413106333696,0,2,1,0,0,Radial Velocity,2007,<a refstr=LIU_ET_AL__2008 href=https://ui.adsa...,2008-01,Ground,Xinglong Station,2.16 m Telescope,Coude Echelle Spectrograph,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Published Confirmed,0.0,<a refstr=KUNITOMO_ET_AL__2011 href=https://ui...,NaN,1.210,NaN,NaN,NaN,NaN,5434.700000,17.100,NaN,NaN,5434.700000,17.100,Msini,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,<a refstr=KUNITOMO_ET_AL__2011 href=https://ui...,NaN,NaN,NaN,2.60,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<a refstr=STASSUN_ET_AL__2019 href=https://ui....,12h20m42.91s,185.178779,+17d47m35.71s,17.793252,78.28058,264.13775,18.33392,177.41790,140.383627,-109.24100,88.1701,93.1846,10.71040,5.726,4.72307,2.943,2.484,2.282,NaN,NaN,NaN,NaN,NaN,0.639,0.732,2.358,2.270,4.44038,NaN,3.83790,NaN,7/23/2014,2011-08,7/23/2014,2.0,1.0,2.0,0.0,0.0,0.0,0.0
2,3,11 Com b,11 Com,b,HD 107383,HIP 60202,TIC 72437047,Gaia DR2 3946945413106333696,0,2,1,0,0,Radial Velocity,2007,<a refstr=LIU_ET_AL__2008 href=https://ui.adsa...,2008-01,Ground,Xinglong Station,2.16 m Telescope,Coude Echelle Spectrograph,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Published Confirmed,0.0,<a refstr=LIU_ET_AL__2008 href=https://ui.adsa...,326.03,1.290,NaN,NaN,NaN,NaN,6165.600000,19.400,NaN,NaN,6165.600000,19.400,Msini,NaN,0.231,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,...,94.80,302.80,NaN,NaN,<a refstr=LIU_ET_AL__2008 href=https://ui.adsa...,G8 III,4742.0,19.00,2.70,-0.35,[Fe/H],2.24300,2.31,NaN,NaN,1.2,NaN,NaN,<a refstr=STASSUN_ET_AL__2019 href=https://ui....,12h20m42.91s,185.178779,+17d47m35.71s,17.793252,78.28058,264.13775,18.33392,177.41790,140.383627,-109.24100,88.1701,93.1846,10.71040,5.726,4.72307,2.943,2.484,2.282,NaN,NaN,NaN,NaN,NaN,0.639,0.732,2.358,2.270,4.44038,NaN,3.83790,NaN,5/14/2014,2008-01,5/14/2014,2.0,1.0,


## 3. Feature selection (top ~10, presence-checked)

We aim for 10 especially relevant predictors. The PS table field names vary across versions; this section
checks for common aliases.

1. `st_met` — stellar metallicity  
2. `st_teff` — stellar effective temperature (K)  
3. `st_mass` — stellar mass (solar)  
4. `pl_rade` — planet radius (Earth radii)  
5. `pl_bmasse` — planet mass (Earth masses)  
6. `pl_dens` or derived density (mass/volume) if possible  
7. `pl_orbper` — orbital period (days)  
8. `pl_orbsmax` — semi-major axis (AU)  
9. `pl_eqt` — planetary equilibrium temperature (K)  
10. `pl_insol` — incident stellar flux (Earth=1)


In [6]:

# Identify available columns or acceptable aliases
colmap = {
    'st_met': ['st_met', 'stellar_metallicity'],
    'st_teff': ['st_teff', 'stellar_teff', 'st_teff_k'],
    'st_mass': ['st_mass', 'stellar_mass', 'sy_msun'],
    'pl_rade': ['pl_rade', 'planet_radius_earth', 'pl_rade_e'],
    'pl_bmasse': ['pl_bmasse', 'pl_masse', 'planet_mass_earth'],
    'pl_dens': ['pl_dens', 'planet_density', 'pl_dens_earth'],
    'pl_orbper': ['pl_orbper', 'orbital_period', 'pl_orbper_days'],
    'pl_orbsmax': ['pl_orbsmax', 'semi_major_axis', 'pl_orbsmax_au'],
    'pl_eqt': ['pl_eqt', 'equilibrium_temp', 'pl_eqt_k'],
    'pl_insol': ['pl_insol', 'insolation', 'pl_insol_earth']
}

selected_cols = {}
for k, candidates in colmap.items():
    c = safe_col(df, candidates)
    if c is not None:
        selected_cols[k] = c

feature_cols = list(selected_cols.values())

print("Selected/derived feature columns found in PS.csv:")
feature_cols


<class 'NameError'>: name 'safe_col' is not defined


## 4. Build a Solar System reference set (targets we **know**)

We create a compact Solar System table with:
- **Features** compatible with PS columns (same star for all: the Sun).
- **Targets**: mineral **group** proportions (coarse, for prototype):  
  - `iron_metal` (core/metal content proxy)  
  - `silicates` (mantle/crust rock-formers)  
  - `water_ice` (surface/subsurface ice fraction)  
  - `sulfates_carbonates` (evaporites/alteration minerals as a proxy for geochemical cycling)

> **Note**: Values below are *reasonable prototyping estimates* consolidated from well-known planetary science references.  
> Use them as placeholders; you should refine/replace with a vetted table when ready.


In [ ]:

# Shared solar values (Sun)
SUN = {
    'st_met': 0.0,         # solar metallicity relative to Sun (by definition ~0 dex)
    'st_teff': 5772.0,     # K
    'st_mass': 1.0         # Msun
}

# Helper to estimate density in Earth units if mass & radius provided (both Earth units)
def dens_earth_units(m_e, r_e):
    if m_e is None or r_e is None or r_e == 0:
        return None
    return m_e / (r_e ** 3)

# Solar System bodies with reasonably known bulk characteristics
# radius (Earth radii), mass (Earth masses), orbital period (days), sma (AU), eqt (approx K), insol (Earth=1)
solsys_rows = [
    # name, rade, bmasse, orbper, sma, eqt, insol, targets: [iron_metal, silicates, water_ice, sulf/carbs]
    ("Mercury", 0.383, 0.055, 87.97, 0.387, 440, 6.67, [0.65, 0.35, 0.00, 0.00]),
    ("Venus",   0.949, 0.815, 224.70, 0.723, 737, 1.91, [0.32, 0.68, 0.00, 0.00]),
    ("Earth",   1.000, 1.000, 365.25, 1.000, 255, 1.00, [0.32, 0.68, 0.00, 0.05]),
    ("Moon",    0.273, 0.0123, 27.32, 0.00257, 220, 1.00, [0.03, 0.97, 0.00, 0.00]),
    ("Mars",    0.532, 0.107, 686.98, 1.524, 210, 0.43, [0.25, 0.70, 0.02, 0.03]),
    ("Ceres",   0.074, 0.00015, 1680.0, 2.77, 160, 0.13, [0.00, 0.40, 0.60, 0.00]),
    ("Io",      0.286, 0.015, 1.769, 0.00282, 110, 1.00, [0.30, 0.60, 0.00, 0.10]),
    ("Europa",  0.245, 0.008, 3.551, 0.00449, 102, 1.00, [0.10, 0.20, 0.70, 0.00]),
    ("Ganymede",0.413, 0.025, 7.155, 0.00716, 110, 1.00, [0.10, 0.30, 0.60, 0.00]),
    ("Callisto",0.378, 0.018, 16.689, 0.0126, 134, 1.00, [0.10, 0.20, 0.70, 0.00]),
    ("Titan",   0.404, 0.0225, 15.945, 0.00817, 94, 1.00, [0.05, 0.15, 0.75, 0.05]),
]

sol_df = pd.DataFrame(solsys_rows, columns=[
    "name","pl_rade","pl_bmasse","pl_orbper","pl_orbsmax","pl_eqt","pl_insol",
    "target_vec"
])

# Add stellar columns
sol_df['st_met'] = SUN['st_met']
sol_df['st_teff'] = SUN['st_teff']
sol_df['st_mass'] = SUN['st_mass']

# Derived density in Earth units
sol_df['pl_dens'] = [dens_earth_units(m, r) for r, m in zip(sol_df['pl_rade'], sol_df['pl_bmasse'])]

# Unpack vectors into columns
target_names = ['iron_metal','silicates','water_ice','sulfates_carbonates']
targets = np.vstack(sol_df['target_vec'].values)
for i, tname in enumerate(target_names):
    sol_df[tname] = targets[:, i]

# Reorder columns for clarity
sol_feature_cols = ['st_met','st_teff','st_mass','pl_rade','pl_bmasse','pl_dens','pl_orbper','pl_orbsmax','pl_eqt','pl_insol']
sol_df = sol_df[['name'] + sol_feature_cols + target_names]
sol_df.head(12)



## 5. Aligning features and building the supervised model

- We align the Solar System features to the feature set discovered in `PS.csv`.
- We train a **regularized linear model** (Ridge inside a MultiOutputRegressor) to learn weights from features → mineral groups.
- We use **Leave-One-Out CV** due to the small sample size to sanity-check generalization.


In [ ]:

# Columns actually available in PS.csv (discovered earlier)
ps_feature_cols = feature_cols

# Solar System training X with only those columns (fallback: drop columns not present in PS)
train_feature_cols = []
for key, candidates in {
    'st_met': ['st_met'],
    'st_teff': ['st_teff'],
    'st_mass': ['st_mass'],
    'pl_rade': ['pl_rade'],
    'pl_bmasse': ['pl_bmasse'],
    'pl_dens': ['pl_dens'],
    'pl_orbper': ['pl_orbper'],
    'pl_orbsmax': ['pl_orbsmax'],
    'pl_eqt': ['pl_eqt'],
    'pl_insol': ['pl_insol']
}.items():
    for c in candidates:
        if c in ps_feature_cols and c in sol_df.columns:
            train_feature_cols.append(c)
            break

X_train = sol_df[train_feature_cols].copy()
Y_train = sol_df[['iron_metal','silicates','water_ice','sulfates_carbonates']].copy()

# Pipeline: impute -> scale -> Ridge (multi-output)
# The following code was developed with the aid of ChatGPT
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocess = ColumnTransformer(
    transformers=[('num', numeric_transformer, train_feature_cols)],
    remainder='drop'
)

base = Ridge(alpha=1.0, random_state=42)
model = Pipeline(steps=[('prep', preprocess),
                       ('reg', MultiOutputRegressor(base))])

# Leave-One-Out CV to get quick MAE scores
loo = LeaveOneOut()
mae_scores = []
pred_vs_true = []

for train_idx, test_idx in loo.split(X_train):
    model.fit(X_train.iloc[train_idx], Y_train.iloc[train_idx])
    pred = model.predict(X_train.iloc[test_idx])
    mae = np.mean(np.abs(pred - Y_train.iloc[test_idx].values))
    mae_scores.append(mae)
    pred_vs_true.append((sol_df.iloc[test_idx]['name'].values[0], pred[0], Y_train.iloc[test_idx].values[0]))

cv_mae = np.mean(mae_scores)
cv_mae


In [ ]:
# Fit on all Solar System data
model.fit(X_train, Y_train)



## 6. Apply the trained model to exoplanets

This produces **predicted mineral group proportions** for each exoplanet with available features and saves a results CSV.



## 6. Apply the trained model to exoplanets

This produces **predicted mineral group proportions** for each exoplanet with available features and saves a results CSV.


In [ ]:

# Prepare exoplanet feature matrix using the same columns
X_exo = df[train_feature_cols].copy()

# Predict (will impute missing values as configured)
Y_pred = model.predict(X_exo)

pred_cols = [f'pred_{t}' for t in ['iron_metal','silicates','water_ice','sulfates_carbonates']]
pred_df = pd.DataFrame(Y_pred, columns=pred_cols)

# Clip to [0,1] and renormalize rows (optional, since regression can output outside range)
pred_df = pred_df.clip(lower=0.0, upper=1.0)
row_sums = pred_df.sum(axis=1).replace(0, np.nan)
pred_df_norm = pred_df.div(row_sums, axis=0).fillna(0.0)

# Save
out_path = Path('exominerals_predictions.csv')
out[out.columns.intersection(list(df.columns) + list(pred_df_norm.columns) + ['ExoMineralIndex'])].to_csv(out_path, index=False)
out_path.as_posix()



## 7. Depot Readiness Score

- Calculates based on distance and availability of materials the readiness for a planet to be turned into a depot that can be used as a rest point before travelling to a final destination


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- 0) Load predictions (produced earlier) ---
pred_path = "exominerals_predictions.csv"
exo = pd.read_csv(pred_path)

# --- 1) Column hygiene / presence-checks ---
# System name (common fields in PS tables): try hostname -> sy_name -> pl_hostname -> fallback to star name-like
sys_col_candidates = ['hostname','sy_name','pl_hostname','hd_name','tic_id','gaia_id']
sys_col = next((c for c in sys_col_candidates if c in exo.columns))

# Distance in parsecs (often 'sy_dist' in PS tables). Optional.
dist_col_candidates = ['sy_dist','st_dist','distance_pc']
dist_col = next((c for c in dist_col_candidates if c in exo.columns))

# Planet identifier (optional; used for tie-breaking / labeling)
pl_name_col = next((c for c in ['pl_name','planet_name','pl_name_clean'] if c in exo.columns))

# Predicted mineral columns (from your pipeline)
pred_cols = ['pred_iron_metal','pred_silicates','pred_water_ice','pred_sulfates_carbonates']
have_all_preds = all(c in exo.columns for c in pred_cols)

# Feature columns used by the model (for a crude confidence proxy)
model_feature_cols = [c for c in ['st_met','st_teff','st_mass','pl_rade','pl_bmasse','pl_dens','pl_orbper','pl_orbsmax','pl_eqt','pl_insol']
                      if c in exo.columns]

# --- 2) Build system-level resource portfolio ---
# We aggregate planet predictions into a system portfolio with two views:
#   - 'max' view catches "best single pit stop" in the system (e.g., icy moon)
#   - 'mean' view captures overall richness spread across planets
agg_max = exo.groupby(sys_col)[pred_cols].max().add_suffix('_max')
agg_mean = exo.groupby(sys_col)[pred_cols].mean().add_suffix('_mean')
counts = exo.groupby(sys_col).size().rename('n_planets')

sys_df = pd.concat([agg_max, agg_mean, counts], axis=1).reset_index()

# Attach distance if we have it (mean distance per system; they should be identical within a system)
if dist_col:
    sys_dist = exo.groupby(sys_col)[dist_col].median().rename('distance_pc')
    sys_df = sys_df.merge(sys_dist, on=sys_col, how='left')

# --- 3) Completeness ---
#   - per-planet feature completeness ratio, then average per system
def row_completeness(row):
    if not model_feature_cols:
        return np.nan
    return row[model_feature_cols].notna().mean()

if model_feature_cols:
    completeness = exo.copy()
    completeness['feat_completeness'] = completeness.apply(row_completeness, axis=1)
    sys_conf = completeness.groupby(sys_col)['feat_completeness'].mean().rename('feat_completeness_mean')
    sys_df = sys_df.merge(sys_conf, on=sys_col, how='left')
else:
    sys_df['feat_completeness_mean'] = np.nan

# Creation of variance columns for predicted values
for p in pred_cols:
    var_name = p + '_var'
    sys_df[var_name] = exo.groupby(sys_col)[p].var().reindex(sys_df[sys_col]).values

# --- 4) Depot Readiness Score (tunable heuristic) ---
# Intuition:
#   - Metals + Silicates are primary for structures; Water/Ice crucial for propellant/life support.
#   - Prefer systems where at least one body is very good (MAX view), but also reward overall richness (MEAN view).
#   - Penalize long distance a bit (if available) and low data completeness.
#   - Penalize high disagreement (variance) across planets (less predictable).
w = {
    'iron_max': 0.25, 'sil_max': 0.25, 'ice_max': 0.15, 'evap_max': 0.05,
    'iron_mean': 0.12, 'sil_mean': 0.12, 'ice_mean': 0.04, 'evap_mean': 0.02
}
# Distance penalty scale: soft; normalize by percentile to avoid killing far systems when distance exists
if 'distance_pc' in sys_df.columns:
    dist = sys_df['distance_pc']
    dist_norm = (dist - dist.quantile(0.1)) / (dist.quantile(0.9) - dist.quantile(0.1) + 1e-9)
    dist_penalty = 0.15 * dist_norm.clip(0, 1)  # up to 0.15 penalty
else:
    dist_penalty = 0.0

# Variance penalty: average of per-pred variance (higher variance → more disagreement across planets)
var_cols = [c for c in sys_df.columns if c.endswith('_var')]

var_norm = sys_df[var_cols].mean(axis=1)
# scale to [0,1] by 90th percentile to avoid outliers dominating
scale = var_norm.quantile(0.9) or 1.0
var_penalty = 0.10 * (var_norm / (scale + 1e-9)).clip(0, 1)

# Completeness bonus: encourage systems with better feature coverage
comp = sys_df['feat_completeness_mean'].fillna(sys_df['feat_completeness_mean'].median())
comp_bonus = 0.10 * (comp.clip(0,1))  # up to +0.10

sys_df['DepotReadinessScore'] = (
    w['iron_max'] * sys_df['pred_iron_metal_max'] +
    w['sil_max']  * sys_df['pred_silicates_max'] +
    w['ice_max']  * sys_df['pred_water_ice_max'] +
    w['evap_max'] * sys_df['pred_sulfates_carbonates_max'] +
    w['iron_mean']* sys_df['pred_iron_metal_mean'] +
    w['sil_mean'] * sys_df['pred_silicates_mean'] +
    w['ice_mean'] * sys_df['pred_water_ice_mean'] +
    w['evap_mean']* sys_df['pred_sulfates_carbonates_mean'] +
    comp_bonus -
    (dist_penalty if isinstance(dist_penalty, (int,float)) else dist_penalty) -
    (var_penalty if isinstance(var_penalty, (int,float)) else var_penalty)
)

# --- 5) Rank & export ---
sys_df = sys_df.sort_values('DepotReadinessScore', ascending=False)

topN = 25
top_systems = sys_df.head(topN).copy()
csv_out = "system_candidates.csv"
sys_df.to_csv(csv_out, index=False)
print(f"Saved all systems ranked to: {csv_out}")
top_systems.head(25)
